# Probing pipeline
Runs data creation, training, evaluation, visualisation and control experiments end to end.

## Imports

In [ ]:
!pip install -r requirements.txt

In [ ]:
import glob
import os
import shutil
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download
from transformer_lens import HookedTransformer

from config import RunConfig
from utils import (
    build_classifiers,
    create_results_path,
    load_trained_clfs,
    save_clf_with_skops,
    save_dataset_locally,
    split_by_layer,
    upload_repo_to_hf,
)
from probes_dataset_creation_script import (
    create_canonical_dataset,
    distractor_word_data,
    neutral_filler_data,
    no_rule_keyword,
    opposite_statuses_rules,
    #canonical_test_heldout_split
)
from train_probes import training
from evaluate_probes import evaluate
from plot_probes import (
    accuracies_from_evaluation_results,
    plot_accuracy_per_layer,
    plot_auroc_curves,
)
from control_experiments import (
    distractor_control,
    double_rule_control,
    neutral_filler_control,
    no_keyword_control,
    p_value_control,
    train_on_shuffled_labels,
    weights_vs_diff_of_means,
)


## Log in to HuggingFace

In [ ]:
#load_dotenv()
login()


## Config

### English


In [ ]:
BASE_DIR = "/workspace/crosslingual-rule-following"
PROBES_DATA_DIR = f"{BASE_DIR}/canonical/probing/probes_data"

LANGUAGE = "en"
DATASET_NAME = f"rule-following-eval_{LANGUAGE}"
RESULTS_FOLDER = f"{BASE_DIR}/canonical/probing/results_{LANGUAGE}"

HF_REPO_IX = "crosslingual-rule-following/canonical-dataset"
HF_REPO_TYPE = "dataset"
MY_HF_REPO_IX = "veerlosar/prism-model-activations"

MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_NAME_LLAMA = "meta-llama/Llama-3.1-8B-Instruct"
HOOK_NAME = "hook_resid_post"
POS_SLICE = -1

CANONICAL_FULL_PATH_IN_REPO = f"{MODEL_NAME_LLAMA}/canonical_full_dataset/{LANGUAGE}"
ACTIVATIONS_IN_HF = f"{CANONICAL_FULL_PATH_IN_REPO}/{HOOK_NAME}_activations.npy"
Y_IN_HF = f"{CANONICAL_FULL_PATH_IN_REPO}/{HOOK_NAME}_labels.npy"

NEUTRAL_PATH_IN_REPO = f"{MODEL_NAME_LLAMA}/neutral_dataset/{LANGUAGE}"
NEUTRAL_ACTIVATIONS_IN_HF = f"{NEUTRAL_PATH_IN_REPO}/{HOOK_NAME}_activations.npy"
NEUTRAL_Y_IN_HF = f"{NEUTRAL_PATH_IN_REPO}/{HOOK_NAME}_labels.npy"

DISTRACTOR_PATH_IN_REPO = f"{MODEL_NAME_LLAMA}/distractor_dataset/{LANGUAGE}"
DISTRACTOR_ACTIVATIONS_IN_HF = f"{DISTRACTOR_PATH_IN_REPO}/{HOOK_NAME}_activations.npy"
DISTRACTOR_Y_IN_HF = f"{DISTRACTOR_PATH_IN_REPO}/{HOOK_NAME}_labels.npy"

NO_KEYWORD_PATH_IN_REPO = f"{MODEL_NAME_LLAMA}/no_keyword_dataset/{LANGUAGE}"
NO_KEYWORD_ACTIVATIONS_IN_HF = f"{NO_KEYWORD_PATH_IN_REPO}/{HOOK_NAME}_activations.npy"
NO_KEYWORD_Y_IN_HF = f"{NO_KEYWORD_PATH_IN_REPO}/{HOOK_NAME}_labels.npy"

DOUBLE_RULE_PATH_IN_REPO = f"{MODEL_NAME_LLAMA}/double_rule_dataset/{LANGUAGE}"
DOUBLE_RULE_ACTIVATIONS_IN_HF = f"{DOUBLE_RULE_PATH_IN_REPO}/{HOOK_NAME}_activations.npy"
DOUBLE_RULE_Y_IN_HF = f"{DOUBLE_RULE_PATH_IN_REPO}/{HOOK_NAME}_labels.npy"

CLASSIFIER_SPEC = {"logistic_regression": {"max_iter": 2000}, "mlp": {}, "knn": {}}
N_PERM = 1000


## Load the model
Used for on-the-fly activation extraction and to read off the layer count.

In [ ]:
from transformers import AutoConfig

In [ ]:
config = AutoConfig.from_pretrained(MODEL_NAME_LLAMA)
config.max_position_embeddings = 8192

In [ ]:
model = HookedTransformer.from_pretrained(MODEL_NAME_LLAMA)
#model2 = HookedTransformer.from_pretrained(MODEL_NAME_LLAMA)
n_layers = model.cfg.n_layers
#n_layers2 = model2.cfg.n_layers

## Run config

In [ ]:
run_cfg = RunConfig(
    language=LANGUAGE,
    n_layers=n_layers,
    dataset_name=DATASET_NAME,
    results_folder=RESULTS_FOLDER,
)
create_results_path(run_cfg)


## Classifiers

In [ ]:
classifiers = build_classifiers(CLASSIFIER_SPEC)


## Data creation
Main train/test/held-out split, pulled from HuggingFace.

In [ ]:
def upload_dataset_activations(dataset_name, x, y, text, my_repo_id):
    local_dir = save_dataset_locally(f"{RESULTS_FOLDER}/upload_staging/{dataset_name}", x, y, text, hook_name=HOOK_NAME)
    upload_repo_to_hf(
        local_dir, run_cfg, repo_type="dataset",
        repo_id=my_repo_id, path_in_repo=f"{MODEL_NAME_LLAMA}/{dataset_name}_dataset/{LANGUAGE}",
    )


In [ ]:
json_data_path = "data/en/test.jsonl"

In [ ]:
dataset = create_canonical_dataset(
    jsonl_in_hf=json_data_path,
    hf_repo_ix=HF_REPO_IX,
    hf_repo_type=HF_REPO_TYPE,
    hf_dataset_repo=MY_HF_REPO_IX,
    model=model,
    hook_name=HOOK_NAME,
    pos_slice=POS_SLICE,
    push_full_dataset_to_hf=True,
    push_path_in_repo=CANONICAL_FULL_PATH_IN_REPO,
    cfg=run_cfg,
)
# run canonical_heldout_split separately or heldout_split depending on the set

Confound datasets, for later control experiments.

In [ ]:
neutral_filler_dataset = neutral_filler_data(
    f"{PROBES_DATA_DIR}/neutral_fillers.json", model, hook_name=HOOK_NAME, pos_slice=POS_SLICE
)
upload_dataset_activations(
    "neutral", neutral_filler_dataset.neutral_x, neutral_filler_dataset.neutral_y, neutral_filler_dataset.neutral_text, MY_HF_REPO_IX
)


In [ ]:
distractor_dataset = distractor_word_data(
    json_data_path, HF_REPO_IX, model, hf_repo_type=HF_REPO_TYPE
)
upload_dataset_activations(
    "distractor", distractor_dataset.distractor_x, distractor_dataset.distractor_y, distractor_dataset.distractor_text, MY_HF_REPO_IX
)


In [ ]:
no_keyword_dataset = no_rule_keyword(
    model, json_data_path, HF_REPO_IX, repo_type=HF_REPO_TYPE, hook_name=HOOK_NAME, pos_slice=POS_SLICE
)
upload_dataset_activations(
    "no_keyword", no_keyword_dataset.nokrule_x, no_keyword_dataset.nokrule_y, no_keyword_dataset.nokrule_text, MY_HF_REPO_IX
)


In [ ]:
double_rule_dataset = opposite_statuses_rules(
    f"{PROBES_DATA_DIR}/double_rule_dataset.json", model, hook_name=HOOK_NAME, pos_slice=POS_SLICE
)
upload_dataset_activations(
    "double_rule", double_rule_dataset.doublerule_x, double_rule_dataset.doublerule_y, double_rule_dataset.doublerule_text, MY_HF_REPO_IX
)


In [ ]:
shutil.rmtree(f"{RESULTS_FOLDER}/upload_staging")


## Training

In [ ]:
#load activations from hf if necessary

In [ ]:
train_X = split_by_layer(dataset.train_x) # util to split tensor into 2D for clf training
trained_classifiers = training(run_cfg, classifiers, train_X, dataset.train_y)


In [ ]:
trained_probes_path = save_clf_with_skops(run_cfg, trained_classifiers)


## Evaluation

In [ ]:
test_X = split_by_layer(dataset.test_x)
validation_evals, validation_eval_path = evaluate(
    run_cfg, trained_classifiers, test_X, dataset.test_y, save_path_prefix="Valid"
)


In [ ]:
held_X = split_by_layer(dataset.held_x)
held_out_evals, held_eval_path = evaluate(
    run_cfg, trained_classifiers, held_X, dataset.held_y, save_path_prefix="Held"
)


## Visualisation

In [ ]:
validation_accuracies, validation_legend = accuracies_from_evaluation_results(validation_evals)
plot_accuracy_per_layer(run_cfg, validation_accuracies, validation_legend, save_path_prefix="Valid")


In [ ]:
held_accuracies, held_legend = accuracies_from_evaluation_results(held_out_evals)
plot_accuracy_per_layer(run_cfg, held_accuracies, held_legend, save_path_prefix="Held")


AUROC curves, one per classifier/layer, on the held-out set.

In [ ]:
y_trues, y_scores, legend = [], [], []
for name, layer_clfs in trained_classifiers.items():
    for layer, clf in layer_clfs.items():
        y_trues.append(dataset.held_y)
        y_scores.append(clf.predict_proba(held_X[layer])[:, 1])
        legend.append(f"{name} layer {layer}")


In [ ]:
plot_auroc_curves(run_cfg, y_trues, y_scores, legend, save_path_prefix="Held")


In [ ]:
upload_repo_to_hf(
    RESULTS_FOLDER, run_cfg, repo_type="dataset",
    repo_id=MY_HF_REPO_IX, path_in_repo=f"{MODEL_NAME_LLAMA}/results_{LANGUAGE}",
)


## Control experiments

In [ ]:
run_cfg = RunConfig(
    language=LANGUAGE, n_layers=n_layers, dataset_name=DATASET_NAME, results_folder=RESULTS_FOLDER,
)


In [ ]:
create_results_path(run_cfg)
classifiers = build_classifiers(CLASSIFIER_SPEC)


In [ ]:
if not glob.glob(f"{run_cfg.trained_probes_path}/*.skops"):
    snapshot_dir = snapshot_download(
        repo_id=MY_HF_REPO_IX, repo_type="dataset",
        allow_patterns=f"{MODEL_NAME_LLAMA}/results_{LANGUAGE}/*",
    )
    shutil.copytree(f"{snapshot_dir}/{MODEL_NAME_LLAMA}/results_{LANGUAGE}", RESULTS_FOLDER, dirs_exist_ok=True)


In [ ]:
trained_probes_path = run_cfg.trained_probes_path


In [ ]:
trained_classifiers = load_trained_clfs(trained_probes_path, run_cfg.language)


In [ ]:
shuffled_results = train_on_shuffled_labels(
    run_cfg, json_data_path, HF_REPO_IX, classifiers,
    hf_repo_type=HF_REPO_TYPE, hf_dataset_repo=MY_HF_REPO_IX,
    activations_in_hf=ACTIVATIONS_IN_HF, y_in_hf=Y_IN_HF,
    model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    push_full_dataset_to_hf=False,
)


In [ ]:
p_value_results = p_value_control(
    run_cfg, json_data_path, HF_REPO_IX, classifiers,
    hf_repo_type=HF_REPO_TYPE, hf_dataset_repo=MY_HF_REPO_IX,
    activations_in_hf=ACTIVATIONS_IN_HF, y_in_hf=Y_IN_HF,
    model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    push_full_dataset_to_hf=False,
    n_perm=N_PERM,
    load_normal_eval_scores=f"{run_cfg.eval_path}/ValidEval_{run_cfg.language}.json",
)


In [ ]:
weights_results = weights_vs_diff_of_means(
    run_cfg, json_data_path, HF_REPO_IX, classifiers,
    hf_repo_type=HF_REPO_TYPE, hf_dataset_repo=MY_HF_REPO_IX,
    activations_in_hf=ACTIVATIONS_IN_HF, y_in_hf=Y_IN_HF,
    model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    push_full_dataset_to_hf=False,
    trained_clfs_folder=trained_probes_path,
)


In [ ]:
neutral_filler_results = neutral_filler_control(
    run_cfg, trained_classifiers, f"{PROBES_DATA_DIR}/neutral_fillers.json",
    model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    hf_dataset_repo=MY_HF_REPO_IX, activations_in_hf=NEUTRAL_ACTIVATIONS_IN_HF, y_in_hf=NEUTRAL_Y_IN_HF,
    n_perm=N_PERM,
)


In [ ]:
distractor_results = distractor_control(
    run_cfg, trained_classifiers, json_data_path, HF_REPO_IX,
    model=model, hf_repo_type=HF_REPO_TYPE, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    hf_dataset_repo=MY_HF_REPO_IX, activations_in_hf=DISTRACTOR_ACTIVATIONS_IN_HF, y_in_hf=DISTRACTOR_Y_IN_HF,
    n_perm=N_PERM,
)


In [ ]:
no_keyword_results = no_keyword_control(
    run_cfg, trained_classifiers, json_data_path, HF_REPO_IX,
    model=model, hf_repo_type=HF_REPO_TYPE, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    hf_dataset_repo=MY_HF_REPO_IX, activations_in_hf=NO_KEYWORD_ACTIVATIONS_IN_HF, y_in_hf=NO_KEYWORD_Y_IN_HF,
    n_perm=N_PERM,
)


In [ ]:
double_rule_results = double_rule_control(
    run_cfg, trained_classifiers, f"{PROBES_DATA_DIR}/double_rule_dataset.json",
    model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    hf_dataset_repo=MY_HF_REPO_IX, activations_in_hf=DOUBLE_RULE_ACTIVATIONS_IN_HF, y_in_hf=DOUBLE_RULE_Y_IN_HF,
    n_perm=N_PERM,
)


## Canonical vs confound: accuracy and AUROC
Compares held-out canonical performance against each confound dataset, per classifier.

In [ ]:
from sklearn.metrics import classification_report

confound_datasets_by_name = {
    "NeutralFiller": (neutral_filler_dataset.neutral_x, neutral_filler_dataset.neutral_y),
    "Distractor": (distractor_dataset.distractor_x, distractor_dataset.distractor_y),
    "NoKeyword": (no_keyword_dataset.nokrule_x, no_keyword_dataset.nokrule_y),
    "DoubleRule": (double_rule_dataset.doublerule_x, double_rule_dataset.doublerule_y),
}


In [ ]:
canonical_accuracies, canonical_legend = accuracies_from_evaluation_results(held_out_evals)
canonical_legend = [f"Canonical-{name}" for name in canonical_legend]

confound_accuracies, confound_legend = [], []
for confound_name, (confound_x, confound_y) in confound_datasets_by_name.items():
    confound_X = split_by_layer(confound_x)
    confound_y = np.array(confound_y)
    for name, layer_clfs in trained_classifiers.items():
        layer_accuracies = {}
        for layer, clf in layer_clfs.items():
            predictions = clf.predict(confound_X[layer])
            layer_accuracies[layer] = classification_report(confound_y, predictions, output_dict=True)["accuracy"]
        confound_accuracies.append(layer_accuracies)
        confound_legend.append(f"{confound_name}-{name}")


In [ ]:
plot_accuracy_per_layer(
    run_cfg, canonical_accuracies + confound_accuracies, canonical_legend + confound_legend,
    save_path_prefix="CanonicalVsConfound",
)


AUROC at each classifier's best canonical held-out layer (all layers x all datasets would be unreadable).

In [ ]:
best_layer_per_classifier = {
    name: max(layer_dict, key=lambda l: layer_dict[l]["accuracy"])
    for name, layer_dict in held_out_evals.items()
}


In [ ]:
canonical_confound_y_trues, canonical_confound_y_scores, canonical_confound_legend = [], [], []
for name, layer_clfs in trained_classifiers.items():
    best_layer = best_layer_per_classifier[name]
    clf = layer_clfs[best_layer]
    canonical_confound_y_trues.append(dataset.held_y)
    canonical_confound_y_scores.append(clf.predict_proba(held_X[best_layer])[:, 1])
    canonical_confound_legend.append(f"Canonical-{name} layer {best_layer}")
    for confound_name, (confound_x, confound_y) in confound_datasets_by_name.items():
        confound_X = split_by_layer(confound_x)
        canonical_confound_y_trues.append(np.array(confound_y))
        canonical_confound_y_scores.append(clf.predict_proba(confound_X[best_layer])[:, 1])
        canonical_confound_legend.append(f"{confound_name}-{name} layer {best_layer}")


In [ ]:
plot_auroc_curves(
    run_cfg, canonical_confound_y_trues, canonical_confound_y_scores, canonical_confound_legend,
    save_path_prefix="CanonicalVsConfound",
)


In [ ]:
upload_repo_to_hf(
    RESULTS_FOLDER, run_cfg, repo_type="dataset",
    repo_id=MY_HF_REPO_IX, path_in_repo=f"{MODEL_NAME_LLAMA}/results_{LANGUAGE}",
)
